# 08 — Caps, Floors & Collars

- **Black cap/floor pricing** with flat and term vol
- **Cap-floor parity**: Cap − Floor = Swap
- **Hull-White** cap pricing
- **AD Greeks**: dCap/dVol, dCap/dRate

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, load_quantlib, compare_table, timed_ms
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np
QL = load_quantlib()

---
## 1. Black Cap/Floor Pricing

A **cap** is a portfolio of caplets. Each caplet pays:

$$
\text{Caplet}_i = N \cdot \tau_i \cdot \max(L_i - K, 0) \cdot P(0, t_i)
$$

Under Black's model, each caplet is a call on the forward LIBOR rate.

In [ ]:
from ql_jax.instruments.capfloor import make_cap, make_floor
from ql_jax.engines.capfloor.black import black_capfloor_price_flat_vol

FLAT_RATE = 0.05
NOTIONAL = 1_000_000.0
START = 2.0 / 365.0
FREQ = 0.25  # quarterly
BLACK_VOL = 0.20

def discount_fn(t):
    return jnp.exp(-FLAT_RATE * t)

def forward_rates(n, start=START, freq=FREQ):
    fwds = []
    for i in range(n):
        t1 = start + i * freq
        t2 = start + (i + 1) * freq
        df1 = jnp.exp(-FLAT_RATE * t1)
        df2 = jnp.exp(-FLAT_RATE * t2)
        fwds.append((df1 / df2 - 1.0) / freq)
    return jnp.array(fwds)

In [ ]:
# Caps at different tenors
strike = 0.04
tenors = [1, 2, 3, 5, 7, 10]

cap_prices = []
floor_prices = []

for tenor in tenors:
    cap = make_cap(strike, START, START + tenor, frequency=FREQ, notional=NOTIONAL)
    flr = make_floor(strike, START, START + tenor, frequency=FREQ, notional=NOTIONAL)
    
    fwds = forward_rates(cap.n_periods)
    cap_pv = float(black_capfloor_price_flat_vol(cap, discount_fn, fwds, BLACK_VOL))
    flr_pv = float(black_capfloor_price_flat_vol(flr, discount_fn, fwds, BLACK_VOL))
    
    cap_prices.append(cap_pv)
    floor_prices.append(flr_pv)
    print(f"  {tenor}Y  Cap={cap_pv:12.2f}  Floor={flr_pv:12.2f}")

---
## 2. Cap-Floor Parity

$$\text{Cap} - \text{Floor} = \text{Swap}_{\text{payer}}$$

In [ ]:
tenor = 5
cap_5y = make_cap(strike, START, START + tenor, frequency=FREQ, notional=NOTIONAL)
flr_5y = make_floor(strike, START, START + tenor, frequency=FREQ, notional=NOTIONAL)
fwds_5y = forward_rates(cap_5y.n_periods)

cap_pv = float(black_capfloor_price_flat_vol(cap_5y, discount_fn, fwds_5y, BLACK_VOL))
flr_pv = float(black_capfloor_price_flat_vol(flr_5y, discount_fn, fwds_5y, BLACK_VOL))

# Swap value: sum of (fwd - strike) * tau * df
swap_pv = 0.0
for i in range(cap_5y.n_periods):
    t_pay = START + (i + 1) * FREQ
    swap_pv += float(NOTIONAL * FREQ * (fwds_5y[i] - strike) * discount_fn(t_pay))

print(f"Cap − Floor    = {cap_pv - flr_pv:12.2f}")
print(f"Payer Swap     = {swap_pv:12.2f}")
print(f"Difference     = {abs(cap_pv - flr_pv - swap_pv):.6f} (should be ≈0)")

---
## 3. AD Greeks: Sensitivity to Vol and Rate

In [ ]:
def cap_price_fn(vol, rate):
    disc = lambda t: jnp.exp(-rate * t)
    cap = make_cap(strike, START, START + 5.0, frequency=FREQ, notional=NOTIONAL)
    fwds = []
    for i in range(cap.n_periods):
        t1 = START + i * FREQ
        t2 = START + (i + 1) * FREQ
        df1 = jnp.exp(-rate * t1)
        df2 = jnp.exp(-rate * t2)
        fwds.append((df1 / df2 - 1.0) / FREQ)
    return black_capfloor_price_flat_vol(cap, disc, jnp.array(fwds), vol)

vol0, rate0 = jnp.float64(BLACK_VOL), jnp.float64(FLAT_RATE)
dvol, drate = jax.grad(cap_price_fn, argnums=(0, 1))(vol0, rate0)

print(f"5Y Cap:")
print(f"  Price        : {float(cap_price_fn(vol0, rate0)):12.2f}")
print(f"  dCap/dVol    : {float(dvol):12.2f}  (vega)")
print(f"  dCap/dRate   : {float(drate):12.2f}  (rho)")

In [ ]:
# Cap price vs vol
import matplotlib.pyplot as plt

vols = jnp.linspace(0.05, 0.50, 50)
prices = [float(cap_price_fn(v, rate0)) for v in vols]
vegas = [float(jax.grad(cap_price_fn, argnums=0)(v, rate0)) for v in vols]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(np.array(vols) * 100, prices, 'b-', linewidth=2)
ax1.set_xlabel('Vol (%)')
ax1.set_ylabel('Cap Price')
ax1.set_title('5Y Cap Price vs Volatility')
ax1.grid(True, alpha=0.3)

ax2.plot(np.array(vols) * 100, vegas, 'r-', linewidth=2)
ax2.set_xlabel('Vol (%)')
ax2.set_ylabel('Vega (dCap/dVol)')
ax2.set_title('Cap Vega via AD')
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

---
## 4. Exercises

1. Price caps with a **term vol structure** (increasing vol for longer caplets).
2. Build a **collar** (long cap + short floor at different strikes) and plot its payoff profile.
3. Use **Hull-White model** cap pricing and compare to Black's model.

---
*End of Notebook 08*